# 08 – Person Anime Works

Esplorazione e data cleaning del dataset `person_anime_works.csv`.

**Colonne:**
| Colonna | Descrizione |
|---|---|
| `person_mal_id` | ID univoco della persona su MyAnimeList |
| `position` | Ruolo svolto dalla persona nell'anime |
| `anime_mal_id` | ID univoco dell'anime su MyAnimeList |

## 1. Import e caricamento dati

In [20]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze
from foreign_key_analyzer import check_fk

df_pw = pd.read_csv('../datasets/person_anime_works.csv')
print(f'Shape: {df_pw.shape}')
print()
df_pw.info()
print()
df_pw.head()

Shape: (458091, 3)

<class 'pandas.DataFrame'>
RangeIndex: 458091 entries, 0 to 458090
Data columns (total 3 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   person_mal_id  458091 non-null  int64
 1   position       458091 non-null  str  
 2   anime_mal_id   458091 non-null  int64
dtypes: int64(2), str(1)
memory usage: 10.5 MB



,person_mal_id,position,anime_mal_id
0,1,Theme Song Performance,3080
1,1,Inserted Song Performance,15699
2,1,Theme Song Performance (OP),247
3,1,Theme Song Performance,258
4,1,Theme Song Performance (ED),34825


**Osservazioni iniziali:**
- Il dataset contiene **458.091 righe** e **3 colonne**.
- Tutte e 3 le colonne sono complete: **nessun valore nullo**.
- I tipi di dati sono adeguati: `int64` per gli ID e `str` per `position`.

## 1.1 Rimozione duplicati esatti

Prima dell'analisi per colonna, rimuoviamo le righe con valori identici in **tutte** le colonne, mantenendo solo la prima occorrenza.

In [21]:
n_originale = len(df_pw)

mask_dup = df_pw.duplicated(keep=False)
n_righe_coinvolte = mask_dup.sum()
n_gruppi = df_pw[mask_dup].duplicated(keep='first').sum()
n_tenute = n_righe_coinvolte - n_gruppi

print(f'Righe totali coinvolte in duplicazioni : {n_righe_coinvolte:,}')
print(f'  → prime occorrenze mantenute         : {n_tenute:,}')
print(f'  → occorrenze extra rimosse           : {n_gruppi:,}')
print()

df_pw.drop_duplicates(keep='first', inplace=True)
print(f'Righe prima della rimozione : {n_originale:,}')
print(f'Righe dopo la rimozione     : {len(df_pw):,}')

Righe totali coinvolte in duplicazioni : 0
  → prime occorrenze mantenute         : 0
  → occorrenze extra rimosse           : 0

Righe prima della rimozione : 458,091
Righe dopo la rimozione     : 458,091


Nessun duplicato esatto trovato. Tutte le 458.091 righe sono già uniche. Il dataset rimane invariato.

Adesso che siamo sicuri che tutte le righe sono uniche, iniziamo l'analisi per colonne utilizzando la nostra libreria `dataset_analyzer`.

## 2. Analisi colonna per colonna

### 2.1 `person_mal_id`

Questa colonna è una **chiave esterna** che referenzia la chiave primaria di `person_details.csv`.

I valori duplicati sono **attesi**: la stessa persona può aver lavorato a più anime in vari ruoli.

I controlli rilevanti sono:
- **Valori nulli**: una chiave esterna nulla indica una riga senza riferimento che va rimossa.
- **Integrità referenziale**: ogni ID presente qui deve esistere in `person_details_clean.csv`.

Usiamo quindi `check_fk` al posto di `analyze`, che effettua entrambi i controlli.

In [22]:
df_persons = pd.read_csv('../datasets_cleaned/person_details_clean.csv')

mask_orphan_person = check_fk(df_pw['person_mal_id'], df_persons['person_mal_id'], child_df=df_pw)

print(f'Null in person_mal_id               : {df_pw["person_mal_id"].isna().sum()}')
print(f'Duplicati in person_mal_id (attesi) : {df_pw["person_mal_id"].duplicated().sum():,}')


  Colonna FK  (tabella figlia)        person_mal_id
  Colonna PK  (tabella padre)         person_mal_id
────────────────────────────────────────────────────────────────────────────────

Riepilogo conteggi
────────────────────────────────────────────────────────────────────────────────

  Righe totali (tabella figlia)       458,091
  Valori null nella FK                0  (0.00%)
  Valori non-null nella FK            458,091  (100.00%)
  Valori unici nella PK padre         75,211

  ✓  Righe con FK valida              458,091  (100.00%)
  ✗  Righe orfane (FK non in PK)      0  (0.00%)
     → ID orfani unici                0

Valutazione integrità referenziale
────────────────────────────────────────────────────────────────────────────────

Nessuna violazione. Tutti i valori FK esistono nella tabella padre.

Null in person_mal_id               : 0
Duplicati in person_mal_id (attesi) : 431,855


**Osservazioni:**
- **Nessun valore nullo**: tutti i record hanno un ID di riferimento valido.
- **Integrità referenziale**: non ci sono righe orfane.

Nessuna pulizia è necessaria.

### 2.2 `position`

Colonna che indica il ruolo svolto dalla persona nell'anime (es. `Director`, `Key Animation`, `Music`). Ci interessa verificare la presenza di valori nulli, duplicati e anomalie. Per questo utilizziamo `analyze`.

I duplicati sono **attesi**: lo stesso ruolo può comparire per persone e anime diversi.

In [23]:
df_pw['position'] = df_pw['position'].str.strip()
analyze(df_pw['position'])


  Nome serie                     position
  dtype                          str
  Dimensione                     458,091
  Range indice                   0 … 458090
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   458,091
  Valori non nulli               458,091  (100.00%)
  Null / NaN                     0  (0.00%)
    di cui stringhe vuote        0  (convertite a NULL)
  Valori unici                   80,036  (17.47%)
  Valori duplicati               378,055  (82.53%)

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  5  caratteri
  Lunghezza max                  573  caratteri
  Lunghezza media                26.92  caratteri
  Lunghezza mediana              21.0000  caratteri
  Dev. standard lunghezza        20.62
  IQR lunghezza                  14.0000

Val

**Osservazioni:**  
- Nessun null. Tutte le righe hanno un valore per la posizione.
- I duplicati sono attesi: lo stesso ruolo può comparire per persone e anime diversi.
- Si nota un numero elevato di valori unici. Questo è atteso in quanto molte posizioni come visto anche tra quelle più lunghe, includono specificazioni per episodio. 

**Nessuna pulizia necessaria.**  

### 2.3 `anime_mal_id`

Questa colonna è una **chiave esterna** che referenzia la chiave primaria `mal_id` di `details.csv`.

I valori duplicati sono **attesi**: ogni anime coinvolge più persone in vari ruoli.

I controlli rilevanti sono:
- **Valori nulli**: una chiave esterna nulla indica una riga senza riferimento che va rimossa.
- **Integrità referenziale**: ogni ID presente qui deve esistere in `details_clean.csv`.

Usiamo quindi `check_fk` al posto di `analyze`, che effettua entrambi i controlli.

In [24]:
df_details = pd.read_csv('../datasets_cleaned/details_clean.csv')

mask_orphan_anime = check_fk(df_pw['anime_mal_id'], df_details['mal_id'], child_df=df_pw)

print(f'Null in anime_mal_id               : {df_pw["anime_mal_id"].isna().sum()}')
print(f'Duplicati in anime_mal_id (attesi) : {df_pw["anime_mal_id"].duplicated().sum():,}')


  Colonna FK  (tabella figlia)        anime_mal_id
  Colonna PK  (tabella padre)         mal_id
────────────────────────────────────────────────────────────────────────────────

Riepilogo conteggi
────────────────────────────────────────────────────────────────────────────────

  Righe totali (tabella figlia)       458,091
  Valori null nella FK                0  (0.00%)
  Valori non-null nella FK            458,091  (100.00%)
  Valori unici nella PK padre         28,955

  ✓  Righe con FK valida              457,330  (99.83%)
  ✗  Righe orfane (FK non in PK)      761  (0.17%)
     → ID orfani unici                66

Valutazione integrità referenziale
────────────────────────────────────────────────────────────────────────────────

761 riga/e (0.17%) violano l'integrità referenziale.

ID orfani
────────────────────────────────────────────────────────────────────────────────

  6076, 6408, 7669, 8481, 20707, 33363, 34595, 35025, 35405, 36734, 38045,
  38748, 40554, 40658, 41090, 49668

**Osservazioni:**
- **Nessun valore nullo**: tutti i record hanno un ID anime valido.
- **Integrità referenziale**: sono presenti righe orfane (ID anime non presenti in `details_clean.csv`). Le righe corrispondenti vanno rimosse.

In [25]:
if mask_orphan_anime.any():
    n_orfane = mask_orphan_anime.sum()
    df_pw = df_pw[~mask_orphan_anime].reset_index(drop=True)
    print(f'Righe orfane rimosse : {n_orfane}')
    print(f'Righe rimanenti      : {len(df_pw):,}')
else:
    print('Nessuna riga orfana da rimuovere.')

Righe orfane rimosse : 761
Righe rimanenti      : 457,330


### 2.4 Chiave primaria `(person_mal_id, position, anime_mal_id)`

La chiave primaria naturale è la tripletta `(person_mal_id, position, anime_mal_id)`: verifichiamo che sia univoca dopo la pulizia.

In [26]:
n_pk_dup = df_pw.duplicated(subset=['person_mal_id', 'position', 'anime_mal_id'], keep=False).sum()
print(f'Duplicati su (person_mal_id, position, anime_mal_id): {n_pk_dup}')

if n_pk_dup > 0:
    print('→ Rimozione duplicati sulla chiave primaria...')
    df_pw.drop_duplicates(subset=['person_mal_id', 'position', 'anime_mal_id'], keep='first', inplace=True)
    print(f'Righe dopo la rimozione: {len(df_pw):,}')
else:
    print('→ Chiave primaria già univoca, nessuna operazione richiesta.')

Duplicati su (person_mal_id, position, anime_mal_id): 0
→ Chiave primaria già univoca, nessuna operazione richiesta.


## 3. Riepilogo e Salvataggio
Le operazioni di pulizia sono state effettuate colonna per colonna nella sezione 2. In questa sezione riepiloghiamo il risultato ed effettuiamo il salvataggio del dataset finale.

In [ ]:
print('Riepilogo Dataset Pulito')
print(f'Righe originali      : {n_originale:>10,}')
print(f'Righe dopo cleaning  : {len(df_pw):>10,}')
print(f'Righe rimosse totali : {n_originale - len(df_pw):>10,}')
print()
df_pw.to_csv('../datasets_cleaned/person_anime_works_clean.csv', index=False)
print('Salvato: datasets_cleaned/person_anime_works_clean.csv')

Riepilogo Dataset Pulito
Righe originali      :    458,091
Righe dopo cleaning  :    457,330
Righe rimosse totali :        761

Salvato: datasets_cleaned/person_anime_works_clean.csv
